In [ ]:
import pandas as pd


mobapp_act['calc_date'] = pd.to_datetime(mobapp_act['calc_date'])
subscr_status['subscr_date_act'] = pd.to_datetime(subscr_status['subscr_date_act'])

first_view = (
    mobapp_act[mobapp_act['metric_id'] == 1]
    .groupby('magnit_id', as_index=False)
    .agg(first_view_date=('calc_date', 'min'))
)

subscr_min = (
    subscr_status
    .groupby(['contact_id', 'magnit_id'], as_index=False)
    .agg(subscr_date_act=('subscr_date_act', 'min'))
)

view_subscr = (
    first_view
    .merge(subscr_min, on='magnit_id', how='left')
)

view_subscr['view_without_subscr'] = (
    view_subscr['subscr_date_act'].isna()
    | (view_subscr['subscr_date_act'] > view_subscr['first_view_date'])
)

view_clients = (
    view_subscr[view_subscr['view_without_subscr']]
    [['magnit_id']]
    .drop_duplicates()
)

view_clients['has_view_without_subscr'] = 1

result = (
    client_cohorts
    .merge(view_clients, on='magnit_id', how='left')
)

result['has_view_without_subscr'] = (
    result['has_view_without_subscr']
    .fillna(0)
)

cohort_result = (
    result
    .groupby('campaigns_cnt', as_index=False)
    .agg(
        client_cnt=('client_id', 'nunique'),
        view_cnt=('has_view_without_subscr', 'sum')
    )
)

cohort_result['view_pct'] = (
    cohort_result['view_cnt']
    / cohort_result['client_cnt']
    * 100
).round(2)

cohort_result